# Task #32 (Story #5): Kiểm tra mẫu thủ công, thống kê, lưu dataset cuối

Đọc `data/processed/orders_step3_labeled.csv` (kết quả Task #31), ép kiểu lại cột ngày tháng và `is_delayed` (CSV không giữ dtype), kiểm tra mẫu thủ công vài đơn để xác nhận logic nhãn đúng, thống kê tỷ lệ trễ tổng thể, rồi lưu bản cuối.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/orders_step3_labeled.csv")

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

print(df["is_delayed"].dtype)
df["is_delayed"] = df["is_delayed"].astype("boolean")
print(df["is_delayed"].dtype)
df["is_delayed"].value_counts(dropna=False)

object
boolean


C:\Users\thanh\AppData\Local\Temp\ipykernel_31548\1748637242.py:3: DtypeWarning: Columns (0: payment_has_boleto, 1: payment_has_credit_card, 2: payment_has_debit_card, 3: payment_has_not_defined, 4: payment_has_voucher) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/orders_step3_labeled.csv")


is_delayed
False    88649
True      7827
<NA>      2965
Name: count, dtype: Int64

## Kiểm tra mẫu thủ công

Lấy ngẫu nhiên vài đơn ở mỗi giá trị nhãn, đối chiếu `order_delivered_customer_date` vs `order_estimated_delivery_date` bằng mắt để xác nhận logic đúng.

In [2]:
sample_cols = ["order_id", "order_status", "order_delivered_customer_date", "order_estimated_delivery_date", "is_delayed"]

for label, group in [(True, "TRỄ"), (False, "ĐÚNG HẠN"), (pd.NA, "CHƯA XÁC ĐỊNH")]:
    print(f"--- {group} (is_delayed={label}) ---")
    mask = df["is_delayed"].isna() if pd.isna(label) else df["is_delayed"] == label
    display(df.loc[mask, sample_cols].sample(3, random_state=42))

--- TRỄ (is_delayed=True) ---


,order_id,order_status,order_delivered_customer_date,order_estimated_delivery_date,is_delayed
14176,91c4fb2a013280c780ea608101fcac7c,delivered,2017-11-18 14:38:35,2017-11-17,True
80150,51d52f115820f57bd947308b77f392db,delivered,2018-01-09 17:19:36,2018-01-05,True
33355,18b5dc19d0cf40a8d307902c30296340,delivered,2018-03-27 22:25:49,2018-03-23,True


--- ĐÚNG HẠN (is_delayed=False) ---


,order_id,order_status,order_delivered_customer_date,order_estimated_delivery_date,is_delayed
71364,ec6dc3377849564c30896b7543229e2d,delivered,2017-11-13 20:33:19,2017-11-21,False
59221,31725408e39733c6fbccf9559d25f0e4,delivered,2017-11-09 19:40:20,2017-11-22,False
4274,fa145f6e474f19a183553a249dc5a73b,delivered,2017-10-27 21:28:01,2017-11-09,False


--- CHƯA XÁC ĐỊNH (is_delayed=<NA>) ---


,order_id,order_status,order_delivered_customer_date,order_estimated_delivery_date,is_delayed
96741,ee6c0c64fe661230879a8491ce7dac76,canceled,NaT,2017-11-22,<NA>
69457,aa380313c19905dd1651bd21e75f09bd,shipped,NaT,2017-08-14,<NA>
91415,5bd293c86326611e6604399e535f1b2b,shipped,NaT,2018-07-19,<NA>


## Thống kê tỷ lệ trễ tổng thể

In [3]:
n_total = len(df)
n_delayed = (df["is_delayed"] == True).sum()
n_on_time = (df["is_delayed"] == False).sum()
n_unknown = df["is_delayed"].isna().sum()

print(f"Tổng số đơn: {n_total}")
print(f"Trễ: {n_delayed} ({n_delayed / n_total:.2%} trên tổng, {n_delayed / (n_delayed + n_on_time):.2%} trên số đơn xác định được)")
print(f"Đúng hạn: {n_on_time} ({n_on_time / n_total:.2%} trên tổng)")
print(f"Chưa xác định: {n_unknown} ({n_unknown / n_total:.2%} trên tổng)")

Tổng số đơn: 99441
Trễ: 7827 (7.87% trên tổng, 8.11% trên số đơn xác định được)
Đúng hạn: 88649 (89.15% trên tổng)
Chưa xác định: 2965 (2.98% trên tổng)


## Lưu dataset cuối

In [4]:
df.to_csv("../data/processed/orders_labeled.csv", index=False)
print("Đã lưu data/processed/orders_labeled.csv:", df.shape)

Đã lưu data/processed/orders_labeled.csv: (99441, 42)
